# IoT Sensor Data Analysis

## Problem Statement

The `SDA_sensor_data` table contains IoT sensor readings. Each reading includes a measurement value and a measurement time stored as a string in the format `MM/DD/YYYY HH:MM:SS`.

For each calendar day, determine the sequence of readings based on measurement time and produce daily summaries for odd and even positions.

## Input Table

### SDA_sensor_data

| Column Name | Data Type |
|------------|-----------|
| measurement_id | INT |
| measurement_value | DECIMAL |
| measurement_time | VARCHAR |

## Requirements

- A reading belongs to the calendar day represented by the date portion of `measurement_time`.
- Order readings within each day by measurement time.
- Number readings sequentially starting at 1 for each day.
- Separate measurements into odd-numbered and even-numbered positions.
- Return one row per day.
- Output `measurement_day` in the format `MM/DD/YYYY 00:00:00`.
- Return results sorted by `measurement_day`.

## Output Columns

| Column Name |
|------------|
| measurement_day |
| odd_sum |
| even_sum |

## Sample Input

### SDA_sensor_data

| measurement_id | measurement_value | measurement_time |
|---------------|------------------|------------------|
| 131233 | 1109.51 | 07/10/2022 09:00:00 |
| 135211 | 1662.74 | 07/10/2022 11:00:00 |
| 523542 | 1246.24 | 07/10/2022 13:15:00 |
| 143562 | 1124.50 | 07/11/2022 15:00:00 |
| 346462 | 1234.14 | 07/11/2022 16:45:00 |

## Sample Output

| measurement_day | odd_sum | even_sum |
|----------------|---------|----------|
| 07/10/2022 00:00:00 | 2355.75 | 1662.74 |
| 07/11/2022 00:00:00 | 1124.50 | 1234.14 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| measurement_day | STRING |
| odd_sum | DECIMAL |
| even_sum | DECIMAL |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from decimal import Decimal
from pyspark.sql.window import Window

SDA_sensor_data_schema = StructType([
    StructField("measurement_id", IntegerType(), True),
    StructField("measurement_value", DecimalType(10, 2), True),
    StructField("measurement_time", StringType(), True)
])

SDA_sensor_data_data = [
    (131233, Decimal('1109.51'), "07/10/2022 09:00:00"),
    (135211, Decimal('1662.74'), "07/10/2022 11:00:00"),
    (523542, Decimal('1246.24'), "07/10/2022 13:15:00"),
    (143562, Decimal('1124.50'), "07/11/2022 15:00:00"),
    (346462, Decimal('1234.14'), "07/11/2022 16:45:00")
]

SDA_sensor_data_df = spark.createDataFrame(
    SDA_sensor_data_data,
    schema=SDA_sensor_data_schema
)

In [0]:
result_df = (
    SDA_sensor_data_df.withColumn(
        "measurement_day", to_date("measurement_time", "MM/dd/yyyy HH:mm:ss")
    )
    .withColumn(
        "rnk",
        rank().over(Window.partitionBy("measurement_day").orderBy("measurement_time")),
    )
    .groupBy("measurement_day")
    .agg(
        sum(when(col("rnk") % 2 == 0, col("measurement_value")).otherwise(0)).alias(
            "even_day_sum"
        ),
        sum(when(col("rnk") % 2 != 0, col("measurement_value")).otherwise(0)).alias(
            "odd_day_sum"
        ),
    )
    .select(
        concat(
            date_format(col("measurement_day"), "MM/dd/yyyy"), lit(" 00:00:00")
        ).alias("measurement_day"),
        col("even_day_sum"),
        col("odd_day_sum"),
    )
)
display(result_df)